# 18. 모델 단일 학습 (Optuna 제외)

**파이프라인 순서**

1. 데이터 로드 & 설정 확인  
2. 앙상블 학습 (단일 실행)  
3. 혼동행렬 (임계값별)  
4. SHAP 중요도 분석  
5. 모델 저장  


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.train_core import (
    AsymmetricSampler, SubsetTrainer, UnderbaggingEnsemble,
    run_training, print_ensemble_summary, plot_subset_prauc,
)
import config.train_config as cfg

print('환경 준비 완료')
print(f'TRAIN_PATH       : {cfg.TRAIN_PATH}')
print(f'VAL_TUNE_PATH    : {cfg.VAL_TUNE_PATH}')
print(f'EVAL_THRESHOLDS  : {cfg.EVAL_THRESHOLDS}')
print(f'REPORT_THRESHOLD : {cfg.REPORT_THRESHOLD}')


환경 준비 완료
TRAIN_PATH       : C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_train.parquet
VAL_TUNE_PATH    : C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\val_tune_raw.parquet
EVAL_THRESHOLDS  : [0.1, 0.2, 0.3, 0.4, 0.5]
REPORT_THRESHOLD : 0.3


## 1. 데이터 로드 & 피처 확인


In [ ]:
df_train    = pd.read_parquet(cfg.TRAIN_PATH)
df_val_tune = pd.read_parquet(cfg.VAL_TUNE_PATH)

_meta = {'serial_number', 'date', 'failure', cfg.TARGET_COL}
FEATURE_COLS = cfg.FEATURE_COLS or [c for c in df_train.columns if c not in _meta]

pos_tr  = df_train[cfg.TARGET_COL].mean()
pos_val = df_val_tune[cfg.TARGET_COL].mean()
print(f'train    : {len(df_train):,} rows  pos_rate={pos_tr:.5f}')
print(f'val_tune : {len(df_val_tune):,} rows  pos_rate={pos_val:.5f}')
print(f'features : {len(FEATURE_COLS)} 개')


## 2. 훈련 및 검증 (단일 실행)


In [ ]:
# Optuna 없이 train_config 파라미터 그대로 단일 훈련
result_single = run_training(
    cfg=cfg,
    feature_cols=FEATURE_COLS,
    run_optuna=False,
    show_plots=True,
)


📂  데이터 로드 중...
   train=291,988  val_tune=8,004,395  features=168
   pos_rate: train=0.09159  val_tune=0.00070

🏋️  최종 앙상블 학습 시작...
⚠️  [Sampler] 정상 행 부족: 필요=267,420 / 보유=265,246 → 보유 전체 사용
✅  [Sampler] 10개 서브셋 생성 완료
   pos=26,742  neg/subset=265,246  total/subset=291,988
  🔧  Subset 1/10 학습 중... 

KeyError: "['s5_diff', 's187_diff', 's198_diff', 'timeout_5s_diff', 's241_diff', 's242_diff', 'total_reads_diff', 'total_seeks_diff', 's183_diff', 's187_14d_sum', 's187_14d_max', 's183_14d_max', 'timeout_5s_14d_sum', 'timeout_5s_14d_max', 'timeout_total_14d_max', 'seek_error_count_14d_sum', 'seek_error_count_14d_max', 's184_14d_max', 's190_14d_max', 's190_14d_mean', 's190_14d_std', 's190_14d_dai', 's190_14d_zscore', 's194_14d_max', 's194_14d_mean', 's194_14d_std', 's194_14d_dai', 's194_14d_zscore', 's241_14d_mean', 's241_14d_dai', 's241_14d_accel', 's242_14d_mean', 's242_14d_dai', 's242_14d_accel', 'total_reads_14d_max', 'total_reads_14d_asfd', 'total_reads_14d_dai', 'total_reads_14d_accel', 'total_seeks_14d_mean', 'total_seeks_14d_dai', 'total_seeks_14d_accel', 's190_14d_ewma', 's194_14d_ewma', 'total_reads_14d_ewma', 's5_28d_max', 's187_28d_sum', 's187_28d_max', 's198_28d_max', 's183_28d_max', 'timeout_5s_28d_sum', 'timeout_5s_28d_max', 'timeout_total_28d_max', 'seek_error_count_28d_

## 3. 혼동행렬 (임계값별)

> 임계값 설정: `train_config.py` → `EVAL_THRESHOLDS`, `REPORT_THRESHOLD`


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_val  = df_val_tune[cfg.TARGET_COL].values
probs  = result_single['ensemble_result'].val_tune_probs

# cfg.EVAL_THRESHOLDS 에서 임계값 목록을 읽어 혼동행렬 시각화
thresholds = cfg.EVAL_THRESHOLDS
fig, axes = plt.subplots(1, len(thresholds), figsize=(4 * len(thresholds), 4))
if len(thresholds) == 1:
    axes = [axes]

for ax, thr in zip(axes, thresholds):
    preds = (probs >= thr).astype(int)
    cm    = confusion_matrix(y_val, preds)
    disp  = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Failure'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Threshold = {thr}', fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrix by Threshold (val_tune)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# cfg.REPORT_THRESHOLD 기준 Classification Report
rpt_thr = cfg.REPORT_THRESHOLD
print(f'\n[Classification Report  threshold={rpt_thr}]')
print(classification_report(y_val, (probs >= rpt_thr).astype(int), target_names=['Normal', 'Failure']))


## 4. SHAP 중요도 분석


In [ ]:
import shap
import warnings
warnings.filterwarnings('ignore')

ensemble_result = result_single['ensemble_result']
feature_cols    = result_single['feature_cols']

X_val_shap = df_val_tune[feature_cols].copy()
SHAP_SAMPLE = min(2000, len(X_val_shap))
X_sample = X_val_shap.sample(SHAP_SAMPLE, random_state=42).reset_index(drop=True)

print(f'SHAP 계산 중... (샘플={SHAP_SAMPLE}, 모델={len(ensemble_result.models)}개)')

sv_list = []
for i, model in enumerate(ensemble_result.models):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample)
    if isinstance(sv, list):
        sv = sv[1]
    sv_list.append(sv)
    print(f'  모델 {i+1}/{len(ensemble_result.models)} 완료', end='\r')

mean_shap = np.mean(sv_list, axis=0)
print('\nSHAP 계산 완료!')


In [ ]:
# SHAP Beeswarm (Summary Plot)
shap.summary_plot(
    mean_shap,
    X_sample,
    feature_names=feature_cols,
    max_display=30,
    show=True,
)


In [ ]:
# SHAP Bar Plot
mean_abs_shap = np.abs(mean_shap).mean(axis=0)
shap_df = pd.DataFrame({'feature': feature_cols, 'importance': mean_abs_shap})
shap_df = shap_df.sort_values('importance', ascending=False).reset_index(drop=True)

TOP_N = 30
fig, ax = plt.subplots(figsize=(10, max(6, TOP_N * 0.3)))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, TOP_N))
top = shap_df.head(TOP_N)
ax.barh(top['feature'][::-1], top['importance'][::-1], color=colors)
ax.set_xlabel('mean(|SHAP value|)', fontsize=11)
ax.set_title(f'SHAP Feature Importance (Top {TOP_N})', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('\n[SHAP 중요도 상위 피처]')
print(shap_df.head(TOP_N).to_string(index=False))


## 5. 모델 및 설정 저장


In [ ]:
import joblib, json
from pathlib import Path

SAVE_DIR = Path(cfg.MODEL_SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for i, model in enumerate(result_single['ensemble_result'].models):
    path = SAVE_DIR / f'subset_{i:02d}.pkl'
    joblib.dump(model, path)
    print(f'  저장: {path}')

feat_path = SAVE_DIR / 'feature_cols.json'
with open(feat_path, 'w', encoding='utf-8') as f:
    json.dump(result_single['feature_cols'], f, ensure_ascii=False, indent=2)
print(f'  피처 목록: {feat_path}')

param_path = SAVE_DIR / 'used_params.json'
with open(param_path, 'w', encoding='utf-8') as f:
    json.dump(result_single['best_params'], f, ensure_ascii=False, indent=2)
print(f'  파라미터 : {param_path}')
